# 🐾 Animal Sound Generator — v17 (GAN)

**Class-conditional GAN with FiLM conditioning.**
Generator starts from pure noise — no shortcut possible.

| Step | Time |
|------|------|
| Mount Drive + unzip data | ~2 min |
| Train GAN (300 epochs) | ~90 min |
| Generate & Download | ~2 min |

### First time: build data zip
Run `colab/build_data.ipynb` once to create `animal1000.zip` on Drive.

In [ ]:
# @title 1. Setup + Mount Drive + Unzip Data
!git clone https://github.com/weseegod/animal_sound_generator.git /content/animal_sound_generator 2>/dev/null
%cd /content/animal_sound_generator
!git pull origin main

!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy matplotlib tqdm soundfile
!mkdir -p models outputs

from google.colab import drive
drive.mount('/content/drive')

import torch, os
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

ZIP = "/content/drive/MyDrive/animal_sound_generator/data/animal1000.zip"
if os.path.exists(ZIP):
    !unzip -qo "{ZIP}" -d data/
    print("✅ Data unzipped")
    for cls in ['Dog','Cat','Chicken','Frog','Bird','Cow']:
        n = len([f for f in os.listdir(f'data/animal1000/{cls}') if f.endswith('.wav')])
        print(f'  {cls}: {n}')
else:
    print("⚠️ animal1000.zip not found. Run build_data.ipynb first.")

In [ ]:
# @title 2. Train GAN (~90 min, 300 epochs)
assert torch.cuda.is_available(), "Enable GPU runtime: Runtime → Change runtime type → L4 GPU"
!python src/gan/train.py

In [ ]:
# @title 3. Generate & Download
!python src/gan/generate.py --n 5 --output-dir outputs/generated

import zipfile, os
with zipfile.ZipFile('v17_animals.zip', 'w') as z:
    for f in os.listdir('outputs/generated'):
        if f.endswith('.wav'):
            z.write(f'outputs/generated/{f}', f)

from google.colab import files
files.download('v16_animals.zip')

DRIVE_MODELS = "/content/drive/MyDrive/animal_sound_generator/models"
!mkdir -p {DRIVE_MODELS}
!cp -v models/gan_generator_best.pth {DRIVE_MODELS}/
print('✅ Checkpoint saved to Drive')